# Conditional Diffusion + GRU: кольцо → сердце

Самый визуально впечатляющий эксперимент серии:

| | |
|---|---|
| $p_0$ | тонкое **кольцо** радиуса 2.0 — идеальная ротационная симметрия |
| $p_1$ | математическое **сердце** — двулопастная кривая с острым основанием |

Почему красиво: нижняя полуплоскость у кольца плотно заполнена, у сердца — практически пуста.  
Плотность буквально **«перетекает вверх»** после разладки.

In [ ]:
import numpy as np
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'device: {device}')

---
## 1. Распределения $p_0$ и $p_1$

**Кольцо** $p_0$: $r = 2.0 + \varepsilon$, $\theta \sim \text{Uniform}[0, 2\pi)$.

**Сердце** $p_1$: классическая параметрическая кривая
$$x(t) = 16 \sin^3 t, \quad y(t) = 13\cos t - 5\cos 2t - 2\cos 3t - \cos 4t$$
масштабированная и центрированная по оси $y$.

In [ ]:
_S       = 0.155   # масштаб сердца
_Y_SHIFT = 0.35    # центровка по y

def sample_p0(n: int) -> torch.Tensor:
    """Тонкое кольцо r ≈ 2.0."""
    theta = torch.rand(n) * 2 * math.pi
    r     = 2.0 + torch.randn(n) * 0.22
    return torch.stack([r * torch.cos(theta), r * torch.sin(theta)], dim=1)

def sample_p1(n: int) -> torch.Tensor:
    """Математическое сердце (равномерно по параметру t)."""
    t  = torch.rand(n) * 2 * math.pi
    x  = 16 * torch.sin(t)**3 * _S
    y  = (13*torch.cos(t) - 5*torch.cos(2*t)
          - 2*torch.cos(3*t) - torch.cos(4*t)) * _S + _Y_SHIFT
    return torch.stack([x, y], dim=1) + torch.randn(n, 2) * 0.12

# --- Визуализация ---
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
for ax, fn, title, c in [
    (axes[0], sample_p0, '$p_0$ — кольцо',   'steelblue'),
    (axes[1], sample_p1, '$p_1$ — сердце',    'crimson'),
]:
    pts = fn(2000)
    ax.scatter(pts[:,0], pts[:,1], s=3, alpha=0.4, color=c)
    ax.set_title(title, fontsize=13)
    ax.set_xlim(-3.5, 3.5); ax.set_ylim(-3.5, 3.5); ax.set_aspect('equal')
    ax.axhline(0, color='gray', lw=0.5, alpha=0.3)
    ax.axvline(0, color='gray', lw=0.5, alpha=0.3)
plt.suptitle('Распределения до и после разладки', fontsize=12)
plt.tight_layout(); plt.show()

---
## 2. Генерация временных рядов

$\tau \sim \text{Uniform}\{1, \ldots, L-1\}$.

In [ ]:
def generate_series(B: int, L: int) -> tuple:
    tau    = torch.randint(1, L, (B,))
    t_idx  = torch.arange(L)[None, :]
    before = (t_idx < tau[:, None]).float()[:, :, None]
    x0_all = sample_p0(B * L).reshape(B, L, 2)
    x1_all = sample_p1(B * L).reshape(B, L, 2)
    return before * x0_all + (1 - before) * x1_all, tau

torch.manual_seed(5)
x_ex, tau_ex = generate_series(1, 256)
x_ex, tau_ex = x_ex[0], tau_ex[0].item()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
ax1.plot(np.arange(256), x_ex[:,0].numpy(), lw=0.8, c='steelblue', alpha=0.8, label='$x_0$')
ax1.plot(np.arange(256), x_ex[:,1].numpy(), lw=0.8, c='orange', alpha=0.8, label='$x_1$')
ax1.axvline(tau_ex, color='red', ls='--', label=f'τ={tau_ex}')
ax1.legend(); ax1.set_xlabel('t'); ax1.set_title('Временной ряд: компоненты')

ax2.scatter(x_ex[:tau_ex,0], x_ex[:tau_ex,1], s=8, c='steelblue', alpha=0.6, label='t<τ (кольцо)')
ax2.scatter(x_ex[tau_ex:,0], x_ex[tau_ex:,1], s=8, c='crimson',   alpha=0.6, label='t≥τ (сердце)')
ax2.set_xlim(-3.5,3.5); ax2.set_ylim(-3.5,3.5); ax2.set_aspect('equal')
ax2.axhline(0,color='gray',alpha=0.3); ax2.axvline(0,color='gray',alpha=0.3)
ax2.legend(); ax2.set_title(f'Точки в пространстве (τ={tau_ex})')
plt.tight_layout(); plt.show()

---
## 3. Архитектура

GRU-кодировщик истории + FiLM-conditional denoiser (идентично предыдущим ноутбукам).

In [ ]:
class NoiseSchedule:
    def __init__(self, T=500, beta_min=1e-4, beta_max=0.02):
        self.T          = T
        self.betas      = torch.linspace(beta_min, beta_max, T)
        self.alphas     = 1 - self.betas
        self.alpha_bars = torch.cumprod(self.alphas, 0)
        self.mus        = self.alpha_bars.sqrt()
        self.sigmas     = (1 - self.alpha_bars).sqrt()

    def get(self, t):
        return self.mus[t, None], self.sigmas[t, None]

    def to(self, dev):
        for a in ('betas','alphas','alpha_bars','mus','sigmas'):
            setattr(self, a, getattr(self, a).to(dev))
        return self


class SinusoidalEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        half  = dim // 2
        freqs = torch.exp(-math.log(10_000) * torch.arange(half) / max(half-1, 1))
        self.register_buffer('freqs', freqs)

    def forward(self, t):
        x = t.float().unsqueeze(-1) * self.freqs
        return torch.cat([x.sin(), x.cos()], dim=-1)


class HistoryEncoder(nn.Module):
    def __init__(self, d_obs=2, hidden=64):
        super().__init__()
        self.gru        = nn.GRU(d_obs, hidden, batch_first=True)
        self.hidden_dim = hidden

    def encode_sequence(self, x):
        B = x.shape[0]
        out, _ = self.gru(x)
        zeros  = torch.zeros(B, 1, self.hidden_dim, device=x.device)
        return torch.cat([zeros, out[:, :-1, :]], dim=1)

    def encode_prefix(self, x):
        if x.shape[1] == 0:
            return torch.zeros(x.shape[0], self.hidden_dim, device=x.device)
        _, h = self.gru(x)
        return h.squeeze(0)


class ConditionalDenoiser(nn.Module):
    def __init__(self, d=2, hidden=128, context_dim=64, n_layers=4):
        super().__init__()
        self.time_emb    = SinusoidalEmbedding(hidden)
        self.input_proj  = nn.Linear(d, hidden)
        self.time_proj   = nn.Linear(hidden, hidden)
        self.film_proj   = nn.Linear(context_dim, 2 * hidden * n_layers)
        self.n_layers    = n_layers
        self.hidden      = hidden
        self.layers      = nn.ModuleList([nn.Linear(hidden, hidden) for _ in range(n_layers)])
        self.output_proj = nn.Linear(hidden, d)

    def forward(self, x, t, ctx):
        h = self.input_proj(x) + self.time_proj(self.time_emb(t))
        film = self.film_proj(ctx)
        scales, shifts = film.chunk(2, dim=-1)
        scales = scales.reshape(-1, self.n_layers, self.hidden)
        shifts = shifts.reshape(-1, self.n_layers, self.hidden)
        for i, layer in enumerate(self.layers):
            h = F.silu(layer(h))
            h = h * (1 + scales[:,i,:]) + shifts[:,i,:]
        return self.output_proj(h)

---
## 4. Обучение

In [ ]:
D            = 2
L            = 256
HIDDEN_ENC   = 64
HIDDEN_DEN   = 128
N_LAYERS     = 4
T_DIFF       = 500
BATCH_SERIES = 32
N_STEPS      = 3000

encoder  = HistoryEncoder(d_obs=D, hidden=HIDDEN_ENC).to(device)
denoiser = ConditionalDenoiser(d=D, hidden=HIDDEN_DEN,
                                context_dim=HIDDEN_ENC, n_layers=N_LAYERS).to(device)
sched    = NoiseSchedule(T_DIFF).to(device)
optimizer = torch.optim.Adam(
    list(encoder.parameters()) + list(denoiser.parameters()), lr=3e-4)

losses = []
for step in range(N_STEPS):
    x, _ = generate_series(BATCH_SERIES, L)
    x     = x.to(device)

    ctx      = encoder.encode_sequence(x)
    B        = x.shape[0]
    x_flat   = x.reshape(B * L, D)
    ctx_flat = ctx.reshape(B * L, HIDDEN_ENC)

    s   = torch.randint(0, T_DIFF, (B * L,), device=device)
    eps = torch.randn(B * L, D, device=device)

    mu_s, sig_s = sched.get(s)
    x_n     = mu_s * x_flat + sig_s * eps
    eps_hat = denoiser(x_n, s, ctx_flat)
    loss    = F.mse_loss(eps_hat, eps)

    optimizer.zero_grad(); loss.backward(); optimizer.step()
    losses.append(loss.item())
    if (step + 1) % 500 == 0:
        print(f'step {step+1:4d}/{N_STEPS}  loss: {np.mean(losses[-100:]):.4f}')

w = 50
plt.figure(figsize=(8, 2.5))
plt.plot(np.convolve(losses, np.ones(w)/w, mode='valid'))
plt.xlabel('step'); plt.ylabel('DSM loss'); plt.title('Кривая обучения')
plt.tight_layout(); plt.show()

---
## 5. Сэмплирование при условии истории

In [ ]:
@torch.no_grad()
def sample_given_history(history: torch.Tensor, n_samples: int = 500) -> torch.Tensor:
    encoder.eval(); denoiser.eval()
    if history.shape[0] == 0:
        ctx = torch.zeros(1, HIDDEN_ENC, device=device)
    else:
        ctx = encoder.encode_prefix(history.unsqueeze(0).to(device))
    ctx = ctx.expand(n_samples, -1)

    x = torch.randn(n_samples, D, device=device)
    for t in reversed(range(sched.T)):
        t_b    = torch.full((n_samples,), t, dtype=torch.long, device=device)
        eps_h  = denoiser(x, t_b, ctx)
        beta_t = sched.betas[t]
        x = (x - beta_t / sched.sigmas[t] * eps_h) / sched.alphas[t].sqrt()
        if t > 0:
            x = x + beta_t.sqrt() * torch.randn_like(x)
    return x.cpu()

---
## 6. Условная генерация: кольцо → сердце

In [ ]:
torch.manual_seed(3)
TAU_TEST  = 90
L_TEST    = 180
full_hist = torch.cat([sample_p0(TAU_TEST), sample_p1(L_TEST - TAU_TEST)])

test_cases = [
    ('пусто\nt=0',           full_hist[:0]),
    ('кольцо\nt=40',         full_hist[:40]),
    ('t=τ-5\nt=85',          full_hist[:85]),
    ('t=τ+5\nt=95',          full_hist[:95]),
    ('сердце\nt=160',        full_hist[:160]),
]

ncols = 2 + len(test_cases)
fig, axes = plt.subplots(1, ncols, figsize=(3.5*ncols, 3.8))

for ax, fn, title, c in [
    (axes[0], sample_p0, 'p₀ (кольцо)',  'steelblue'),
    (axes[1], sample_p1, 'p₁ (сердце)',  'crimson'),
]:
    pts = fn(700)
    ax.scatter(pts[:,0], pts[:,1], s=4, alpha=0.4, color=c)
    ax.set_title(title, fontsize=9)
    ax.set_xlim(-3.5,3.5); ax.set_ylim(-3.5,3.5); ax.set_aspect('equal')
    ax.axhline(0,color='gray',alpha=0.2); ax.axvline(0,color='gray',alpha=0.2)

for ax, (label, hist) in zip(axes[2:], test_cases):
    smp = sample_given_history(hist, n_samples=700)
    ax.scatter(smp[:,0], smp[:,1], s=4, alpha=0.35, color='mediumpurple')
    if len(hist) > 0:
        shown  = hist[-min(20, len(hist)):]
        c_hist = 'steelblue' if len(hist) < TAU_TEST else 'crimson'
        ax.scatter(shown[:,0], shown[:,1], s=18, alpha=0.7, color=c_hist, marker='x', zorder=5)
    ax.set_title(label, fontsize=9)
    ax.set_xlim(-3.5,3.5); ax.set_ylim(-3.5,3.5); ax.set_aspect('equal')
    ax.axhline(0,color='gray',alpha=0.2); ax.axvline(0,color='gray',alpha=0.2)

plt.suptitle(f'Условная генерация (τ={TAU_TEST}): плотность перетекает из кольца в сердце',
             y=1.04, fontsize=10)
plt.tight_layout(); plt.show()

---
## 7. Анимация: плотность перетекает кольцо → сердце

In [ ]:
from scipy.stats import gaussian_kde
import matplotlib.animation as animation
from IPython.display import Image as IPImage

torch.manual_seed(42)
TAU_VIS  = 100
L_VIS    = 200
hist_vis = torch.cat([sample_p0(TAU_VIS), sample_p1(L_VIS - TAU_VIS)])
hist_np  = hist_vis.numpy()

LIM      = 3.5
grid_pts = np.linspace(-LIM, LIM, 90)
X, Y     = np.meshgrid(grid_pts, grid_pts)
pos_grid = np.stack([X.ravel(), Y.ravel()])

print('Computing reference densities…')
ref_p0 = sample_p0(5000).numpy()
ref_p1 = sample_p1(5000).numpy()
Z_p0   = gaussian_kde(ref_p0.T, bw_method=0.12)(pos_grid).reshape(X.shape)
Z_p1   = gaussian_kde(ref_p1.T, bw_method=0.12)(pos_grid).reshape(X.shape)

FRAME_STEP = 8
frames_t   = list(range(0, L_VIS + 1, FRAME_STEP))
N_COND     = 160

print(f'Pre-computing {len(frames_t)} frames…')
cond_samples = {}
for i, t in enumerate(frames_t):
    cond_samples[t] = sample_given_history(hist_vis[:t], n_samples=N_COND).numpy()
    if (i+1) % 5 == 0: print(f'  {i+1}/{len(frames_t)}')
print('Done.')

def _style(ax, title, lim=LIM):
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim); ax.set_aspect('equal')
    ax.axhline(0, c='white', lw=0.5, alpha=0.3)
    ax.axvline(0, c='white', lw=0.5, alpha=0.3)
    ax.tick_params(labelsize=8, colors='white')
    ax.set_facecolor('#0d0d1a')
    [s.set_color('white') for s in ax.spines.values()]
    ax.set_title(title, fontsize=10, pad=4, color='white')

fig = plt.figure(figsize=(17, 9), facecolor='#0d0d1a')
gs  = fig.add_gridspec(2, 3, height_ratios=[1.3, 0.9], hspace=0.45, wspace=0.3)
ax_true = fig.add_subplot(gs[0,0])
ax_kde  = fig.add_subplot(gs[0,1])
ax_cond = fig.add_subplot(gs[0,2])
ax_ts   = fig.add_subplot(gs[1,:])
ax_ts.set_facecolor('#0d0d1a')
for s in ax_ts.spines.values(): s.set_color('white')
ax_ts.tick_params(colors='white')
ax_ts.xaxis.label.set_color('white')

ax_ts.plot(range(L_VIS), hist_np[:,0], c='cornflowerblue', lw=0.8, alpha=0.85, label='$x_0$')
ax_ts.plot(range(L_VIS), hist_np[:,1], c='gold',           lw=0.8, alpha=0.85, label='$x_1$')
ax_ts.axvline(TAU_VIS, c='salmon', ls='--', lw=1.5, alpha=0.85, label=f'τ={TAU_VIS}')
ax_ts.set_xlim(0, L_VIS-1); ax_ts.set_ylim(-LIM, LIM)
ax_ts.set_xlabel('t', color='white')
leg = ax_ts.legend(fontsize=9, loc='upper right')
for t_ in leg.get_texts(): t_.set_color('white')
ax_ts.set_title(f'Временной ряд  (τ={TAU_VIS})', fontsize=10, color='white')

ptr  = ax_ts.axvline(0, c='white', lw=2, zorder=5, alpha=0.8)
ttxt = ax_ts.text(2, LIM - 0.7, 't = 0', fontsize=11, fontweight='bold', color='white')

CM_TRUE = 'plasma'
CM_KDE  = 'viridis'
CM_COND = 'magma'

def update(fi):
    t   = frames_t[fi]
    ptr.set_xdata([t, t])
    ttxt.set_x(min(t + 3, L_VIS - 28)); ttxt.set_text(f't = {t}')

    Zt  = Z_p0 if t < TAU_VIS else Z_p1
    lbl = 'p₀  кольцо' if t < TAU_VIS else 'p₁  сердце'

    ax_true.cla()
    ax_true.contourf(X, Y, Zt, levels=14, cmap=CM_TRUE)
    ax_true.contour(X, Y, Zt,  levels=6, colors='white', linewidths=0.6, alpha=0.5)
    _style(ax_true, f'Истинная плотность\n{lbl}')

    ax_kde.cla()
    if t >= 6:
        try:
            obs = hist_np[:t]
            Zk  = gaussian_kde(obs.T, bw_method=0.25)(pos_grid).reshape(X.shape)
            ax_kde.contourf(X, Y, Zk, levels=12, cmap=CM_KDE)
            ax_kde.contour(X, Y, Zk,  levels=5, colors='white', linewidths=0.5, alpha=0.45)
            ax_kde.scatter(*obs[-12:].T, s=12, c='lime', alpha=0.6, zorder=5)
        except Exception: pass
    _style(ax_kde, f'KDE наблюдений  (n={t})')

    ax_cond.cla()
    smp = cond_samples[t]
    try:
        Zc = gaussian_kde(smp.T, bw_method=0.28)(pos_grid).reshape(X.shape)
        ax_cond.contourf(X, Y, Zc, levels=12, cmap=CM_COND)
        ax_cond.contour(X, Y, Zc,  levels=5, colors='white', linewidths=0.5, alpha=0.45)
    except Exception: pass
    ax_cond.scatter(*smp.T, s=5, alpha=0.25, c='white', zorder=4)
    _style(ax_cond, 'Условная генерация\np(x | история[:t])')

anim = animation.FuncAnimation(fig, update, frames=len(frames_t), interval=200, blit=False)
anim.save('ring_to_heart.gif', writer='pillow', fps=5, dpi=90)
print('Saved: ring_to_heart.gif')
plt.close(fig)
IPImage('ring_to_heart.gif')

---
## 8. Кривая адаптации

Используем обученный MLP-классификатор: $P(p_1 \mid \text{сэмплы модели})$.  
До разладки ≈ 0 (генерирует кольцо), после — ≈ 1 (генерирует сердце).

In [ ]:
torch.manual_seed(0)
clf_x = torch.cat([sample_p0(3000), sample_p1(3000)])
clf_y = torch.cat([torch.zeros(3000), torch.ones(3000)])

clf = nn.Sequential(
    nn.Linear(2, 64), nn.ReLU(),
    nn.Linear(64, 64), nn.ReLU(),
    nn.Linear(64, 1)
)
clf_opt = torch.optim.Adam(clf.parameters(), lr=1e-3)
for _ in range(1000):
    idx  = torch.randperm(6000)[:256]
    loss = F.binary_cross_entropy_with_logits(clf(clf_x[idx]).squeeze(), clf_y[idx])
    clf_opt.zero_grad(); loss.backward(); clf_opt.step()

with torch.no_grad():
    acc = ((clf(clf_x).squeeze() > 0) == (clf_y > 0.5)).float().mean().item()
print(f'Classifier accuracy: {acc:.3f}')

torch.manual_seed(7)
TAU_CURVE  = 100
hist_curve = torch.cat([sample_p0(TAU_CURVE), sample_p1(150 - TAU_CURVE)])

steps_vis  = list(range(0, 151, 5))
p1_scores  = []
clf.eval()
for t in steps_vis:
    smp = sample_given_history(hist_curve[:t], n_samples=300)
    with torch.no_grad():
        p1_scores.append(torch.sigmoid(clf(smp)).mean().item())

fig, ax = plt.subplots(figsize=(11, 3.5))
ax.set_facecolor('#0d0d1a')
fig.patch.set_facecolor('#0d0d1a')
for sp in ax.spines.values(): sp.set_color('white')
ax.tick_params(colors='white')
ax.xaxis.label.set_color('white')
ax.yaxis.label.set_color('white')
ax.title.set_color('white')

ax.plot(steps_vis, p1_scores, 'o-', color='mediumpurple', ms=5, lw=1.8,
        label='P(p₁ | сэмплы модели)')
ax.fill_between(steps_vis, p1_scores, alpha=0.2, color='mediumpurple')
ax.axvline(TAU_CURVE, color='salmon', ls='--', lw=2, label=f'τ={TAU_CURVE}')
ax.axhline(0.0, color='cornflowerblue', ls=':', alpha=0.6, label='кольцо (≈ 0)')
ax.axhline(1.0, color='crimson',        ls=':', alpha=0.6, label='сердце (≈ 1)')
ax.set_xlabel('Длина истории t'); ax.set_ylabel('P(p₁ | сэмплы)')
ax.set_title('Адаптация: плотность перетекает из кольца в сердце')
leg = ax.legend(fontsize=9)
for t_ in leg.get_texts(): t_.set_color('white')
plt.tight_layout(); plt.show()

---
## 8. Тест на разладку: DDIM-кодирование + MMD против $\mathcal{N}(0,I)$

### Мотивация

Обученная модель $\hat\varepsilon_\theta$ определяет **probability flow ODE** — детерминированную биекцию $\mathcal{T}: \mathbb{R}^d \to \mathbb{R}^d$:

$$X_0 \sim p_0 \implies Z_t = \mathcal{T}(X_t;\,h_{\mathrm{fix}}) \sim \mathcal{N}(0, I)$$
$$X_0 \sim p_1 \implies Z_t \not\sim \mathcal{N}(0, I)$$

Нет оснований считать $\mathcal{T}_*(p_1)$ гауссовским (это биекция над двухмодовым $p_1$, результат тоже двухмодовой). Нужен **непараметрический** тест на соответствие $\mathcal{N}(0,I)$.

---

### DDIM-кодирование (детерминированный прямой проход $x_0 \to Z_T$)

$$x_{k+1} = \mu_{t_{k+1}} \cdot \frac{x_k - \sigma_{t_k}\,\hat\varepsilon_\theta(x_k, t_k, h)}{\mu_{t_k}} + \sigma_{t_{k+1}}\,\hat\varepsilon_\theta(x_k, t_k, h)$$

---

### Maximum Mean Discrepancy (MMD)

MMD² между эмпирической мерой окна $\hat P_w$ и известным $P_0 = \mathcal{N}(0,I)$:

$$\widehat{\mathrm{MMD}}^2 = \underbrace{\frac{1}{w^2}\sum_{i,j} k(Z_i,Z_j)}_{\text{A}} - \underbrace{\frac{2}{w}\sum_i \mathbb{E}_{Y\sim P_0}[k(Z_i,Y)]}_{\text{B}} + \underbrace{\mathbb{E}_{Y,Y'\sim P_0}[k(Y,Y')]}_{\text{C}}$$

Для RBF-ядра $k(x,y) = \exp(-\|x-y\|^2/2\sigma^2)$ члены B и C вычисляются **аналитически** (без сэмплирования):

$$\mathbb{E}_{Y\sim\mathcal{N}(0,I)}[k(z,Y)] = \left(\frac{\sigma^2}{\sigma^2+1}\right)^{d/2} \exp\!\left(-\frac{\|z\|^2}{2(\sigma^2+1)}\right)$$

$$\mathbb{E}_{Y,Y'\sim\mathcal{N}(0,I)}[k(Y,Y')] = \left(\frac{\sigma^2}{\sigma^2+2}\right)^{d/2}$$

Только член A требует пар расстояний внутри окна: $O(w^2)$ операций.

Свойства:
- $\widehat{\mathrm{MMD}}^2 = 0 \iff \hat P_w = P_0$ (в пределе $w\to\infty$)
- Улавливает **все** моменты — не делает предположений о форме $\mathcal{T}_*(p_1)$
- Нулевой порог калибруется **однократно** бутстрэпом из $\mathcal{N}(0,I)$

In [ ]:

N_DDIM = 100   # DDIM шагов (T_DIFF=500, субдискретизация 5×)

@torch.no_grad()
def ddim_encode(x0: torch.Tensor, ctx: torch.Tensor, n_steps: int = N_DDIM) -> torch.Tensor:
    """DDIM-инверсия: x0 (d,) -> z_T (d,). Если x0 ~ p_0, то z_T ≈ N(0, I)."""
    encoder.eval(); denoiser.eval()
    x = x0.clone().float().to(device)

    idx = torch.linspace(0, sched.T - 1, n_steps + 1).long().to(device)

    for k in range(n_steps):
        t_c, t_n = idx[k], idx[k + 1]
        eps = denoiser(x.unsqueeze(0), t_c.view(1), ctx.unsqueeze(0)).squeeze(0)

        mu_c,  sig_c  = sched.mus[t_c],  sched.sigmas[t_c]
        mu_n,  sig_n  = sched.mus[t_n],  sched.sigmas[t_n]

        x0_pred = (x - sig_c * eps) / mu_c   # predicted x_0
        x = mu_n * x0_pred + sig_n * eps      # DDIM forward step

    return x.cpu()


# Фиксированный контекст: прогрев из p_0
torch.manual_seed(0)
N_WARMUP_KL = 30
x_warmup_kl = sample_p0(N_WARMUP_KL).to(device)
with torch.no_grad():
    h_fixed_kl = encoder.encode_prefix(x_warmup_kl.unsqueeze(0)).squeeze(0)  # (HIDDEN_ENC,)

z_test = ddim_encode(sample_p0(1).squeeze(0), h_fixed_kl)
print(f"h_fixed_kl: {h_fixed_kl.shape}")
print(f"Пример z_T из p_0: {z_test.numpy().round(3)}")
print(f"  ||z_T|| = {z_test.norm().item():.3f}  (ожидаем ≈ sqrt({D}) = {D**0.5:.2f} в среднем)")

### Нулевое распределение $\widehat{\mathrm{MMD}}^2$ и калибровка порога

---

#### Шаг 1 — MMD² как вырожденная U-статистика

Перепишем статистику через **центрированное ядро**:

$$\tilde{h}(x,y) = k(x,y) - \mathbb{E}_Y k(x,Y) - \mathbb{E}_Y k(y,Y) + \mathbb{E}_{Y,Y'} k(Y,Y')$$

Тогда несмещённая оценка MMD² — это U-статистика степени 2:

$$\widehat{\mathrm{MMD}}^2_u = \frac{1}{w(w-1)} \sum_{i \neq j} \tilde{h}(z_i, z_j)$$

Ключевое свойство под $H_0$ ($Z \sim P_0$): $\;\mathbb{E}[\tilde{h}(Z, z)] = 0$ для любого фиксированного $z$.

Это называется **вырождением** (degeneracy). Отсюда $\mathbb{E}[\widehat{\mathrm{MMD}}^2_u] = 0$ под $H_0$, а стандартная ЦПТ для U-статистик не работает — нужен другой масштаб.

---

#### Шаг 2 — Разложение Хёффдинга и спектральная теорема

Введём оператор $T_{\tilde h}: L^2(P_0) \to L^2(P_0)$:

$$(T_{\tilde h} f)(x) = \int \tilde{h}(x,y)\,f(y)\,dP_0(y)$$

Поскольку $\tilde{h}$ симметрично и $\mathbb{E}_Y \tilde{h}(Z,Y) = 0$ под $H_0$, оператор $T_{\tilde h}$ компактен и самосопряжён, поэтому имеет спектральное разложение:

$$\tilde{h}(x,y) = \sum_{l=1}^{\infty} \lambda_l \,\phi_l(x)\,\phi_l(y)$$

где $\{\phi_l\}$ — ортонормированный базис в $L^2(P_0)$ (собственные функции), $\mathbb{E}[\phi_l(Z)] = 0$, $\mathrm{Var}[\phi_l(Z)] = 1$.

Подставляя в U-статистику:

$$\frac{1}{w}\sum_{i\neq j} \tilde{h}(z_i,z_j) = \sum_{l=1}^{\infty} \lambda_l \left[\underbrace{\left(\frac{1}{\sqrt{w}}\sum_{i=1}^w \phi_l(z_i)\right)}_{S_l}\right]^2 - \sum_{l=1}^\infty \lambda_l + o_p(1)$$

По ЦПТ каждый $S_l \xrightarrow{d} \mathcal{N}(0,1)$, и $S_l \perp S_m$ для $l \neq m$ (разные собственные функции, разные слагаемые). Итого:

$$\boxed{w \cdot \widehat{\mathrm{MMD}}^2_u \;\xrightarrow{d}\; \sum_{l=1}^{\infty} \lambda_l\,(Z_l^2 - 1), \qquad Z_l \overset{\mathrm{iid}}{\sim} \mathcal{N}(0,1)}$$

---

#### Шаг 3 — Собственные числа для RBF + $\mathcal{N}(0,I)$

Под $P_0 = \mathcal{N}(0,I)$ собственные функции $T_{\tilde h}$ — полиномы Эрмита $He_l(x)$ (ортонормированные относительно гауссовой меры). Собственные числа убывают **геометрически**:

$$\lambda_l \propto \left(\frac{\sigma^2}{\sigma^2 + 2}\right)^l$$

При $\sigma = \sqrt{d} = \sqrt{2}$: $\lambda_l \propto (1/2)^l$. Сумма $\sum_l \lambda_l < \infty$ — оператор компактен класса Гильберта–Шмидта.

Следствие: распределение $\sum_l \lambda_l(Z_l^2-1)$ определяется главным образом **первыми несколькими** слагаемыми. Но замкнутой формулы CDF не существует.

---

#### Шаг 4 — Как порог зависит от ширины окна $w$

Из предельного результата: распределение $w \cdot \widehat{\mathrm{MMD}}^2$ **не зависит от $w$** при больших $w$. Значит:

$$h_{\mathrm{mmd}}(w) \approx \frac{c_\alpha}{w}, \qquad c_\alpha = F^{-1}_{H_0}(1-\alpha)$$

где $F_{H_0}$ — CDF предельного распределения. Порог **убывает как $1/w$**:

| $w$ | $h_{\mathrm{mmd}} \approx$ | Задержка обнаружения | Мощность |
|---|---|---|---|
| $\uparrow$ (большое) | $\downarrow$ мал | $\uparrow$ велика | $\uparrow$ высокая |
| $\downarrow$ (малое) | $\uparrow$ велик | $\downarrow$ мала | $\downarrow$ низкая |

Это фундаментальный трейдофф: большее окно → меньший порог → лучше различаем $p_0$ и $p_1$, но тревога возможна не раньше чем через $w$ шагов после разладки.

---

#### Шаг 5 — Почему Монте-Карло, а не асимптотика

Для конечного $w$ сходимость к предельному распределению только **приближённая**. Монте-Карло:

```python
null_stats = [mmd2_vs_gaussian(rng.standard_normal((W_MMD, D))) for _ in range(3000)]
h_mmd = np.quantile(null_stats, 1 - ALPHA)
```

Это симулирует **точное конечновыборочное** распределение $\widehat{\mathrm{MMD}}^2$ при $Z_i \overset{\mathrm{iid}}{\sim} \mathcal{N}(0,I)$ и данном $w$ — без какого-либо приближения. Асимптотика используется здесь только для понимания структуры, не для вычислений.

In [ ]:
from scipy.spatial.distance import cdist

W_MMD     = 50
ALPHA     = 0.005
SIGMA_MMD = np.sqrt(D)   # sigma ~ sqrt(d): median heuristic for N(0,I)

def mmd2_vs_gaussian(Z_win, sigma=None, d=None):
    """One-sample MMD² vs N(0,I) with RBF kernel. Terms B and C are analytical."""
    if sigma is None: sigma = SIGMA_MMD
    if d     is None: d     = D
    C   = (sigma**2 / (sigma**2 + 2)) ** (d / 2)
    h_B = (sigma**2 / (sigma**2 + 1)) ** (d / 2)
    sq  = cdist(Z_win, Z_win, 'sqeuclidean')
    termA = np.exp(-sq / (2 * sigma**2)).mean()
    termB = 2 * (h_B * np.exp(-np.sum(Z_win**2, 1) / (2*(sigma**2+1)))).mean()
    return termA - termB + C

# ── Верификация DDIM: кодируем N_CAL точек из p0 ─────────────────────────────
torch.manual_seed(42)
N_CAL = 200
Z_cal = np.zeros((N_CAL, D))
for i in range(N_CAL):
    Z_cal[i] = ddim_encode(sample_p0(1).squeeze(0), h_fixed_kl).numpy()

fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))

axes[0].hist(Z_cal[:, 0], bins=40, density=True, alpha=0.6, color='steelblue', label='z₀')
axes[0].hist(Z_cal[:, 1], bins=40, density=True, alpha=0.6, color='tomato',    label='z₁')
xx = np.linspace(-3.5, 3.5, 300)
axes[0].plot(xx, np.exp(-xx**2/2)/np.sqrt(2*np.pi), 'k-', lw=2, label='N(0,1)')
axes[0].set_title('Маргинали DDIM(p₀) — диагностика'); axes[0].legend(fontsize=8)

axes[1].scatter(Z_cal[:, 0], Z_cal[:, 1], s=4, alpha=0.3, color='steelblue')
axes[1].set_title('Латентные коды DDIM(p₀)')
axes[1].set_aspect('equal'); axes[1].set_xlim(-4, 4); axes[1].set_ylim(-4, 4)
axes[1].axhline(0, c='gray', lw=0.5); axes[1].axvline(0, c='gray', lw=0.5)

# ── Калибровка порога: бутстрэп из точной N(0,I) ─────────────────────────────
# Предполагаем DDIM(p0) = N(0,I) (идеально обученный деноизер).
# Нулевые окна берём напрямую из N(0,I) — не тратим точки из потока на калибровку.
print(f"Калибровка по N(0,I) (w={W_MMD}, sigma={SIGMA_MMD:.2f}, n_boot=3000)...")
rng = np.random.default_rng(0)
null_stats = np.array([mmd2_vs_gaussian(rng.standard_normal((W_MMD, D))) for _ in range(3000)])
h_mmd = np.quantile(null_stats, 1 - ALPHA)

axes[2].hist(null_stats, bins=40, density=True, alpha=0.6, color='mediumpurple',
             label='MMD² под H₀  [из N(0,I)]')
axes[2].axvline(h_mmd, c='r', ls='--', lw=2, label=f'q_{{1-α}} = {h_mmd:.5f}')
axes[2].set_title(f'Нулевое распред. MMD²  (w={W_MMD}, σ={SIGMA_MMD:.2f})')
axes[2].legend(fontsize=8)

plt.tight_layout(); plt.show()
print(f"σ = {SIGMA_MMD:.3f},  w = {W_MMD},  α = {ALPHA},  h_mmd = {h_mmd:.6f}")

### Алгоритм обнаружения

На каждом шаге $t$:
1. Закодировать $X_t \xrightarrow{\mathcal{T}} Z_t$ через DDIM-инверсию с $h_{\mathrm{fix}}$
2. Обновить скользящее окно $\{Z_{t-w+1},\ldots,Z_t\}$
3. Вычислить $\widehat{\mathrm{MMD}}^2$ — члены B и C аналитически, член A через попарные расстояния
4. Тревога если $\widehat{\mathrm{MMD}}^2 \geq h_{\mathrm{mmd}}$ (порог из однократной калибровки)

In [ ]:
torch.manual_seed(42)

TAU_DET = 100
L_DET   = 200

x_det = torch.cat([sample_p0(TAU_DET), sample_p1(L_DET - TAU_DET)])

Z_det = np.zeros((L_DET, D))
for i in range(L_DET):
    Z_det[i] = ddim_encode(x_det[i], h_fixed_kl).numpy()

mmd_det = np.full(L_DET, np.nan)
for t in range(W_MMD, L_DET):
    mmd_det[t] = mmd2_vs_gaussian(Z_det[t - W_MMD:t])

alarm_t = next((t for t in range(W_MMD, L_DET) if mmd_det[t] >= h_mmd), None)
delay   = alarm_t - TAU_DET if alarm_t is not None else None

print(f"Порог              h_mmd = {h_mmd:.6f}")
print(f"Истинная разладка  tau   = {TAU_DET}")
print(f"Тревога            tau_hat = {alarm_t}")
print(f"Задержка               = {delay}")

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(13, 9), sharex=True)
t_ax = np.arange(L_DET)

# Panel 1 — observed series
axes[0].plot(t_ax, x_det[:, 0].numpy(), lw=0.7, color='steelblue', label='x[0]', alpha=0.8)
axes[0].plot(t_ax, x_det[:, 1].numpy(), lw=0.7, color='orange',    label='x[1]', alpha=0.8)
axes[0].axvline(TAU_DET, color='red',   ls='--', lw=1.5, label=f'tau={TAU_DET}')
if alarm_t:
    axes[0].axvline(alarm_t, color='green', ls=':', lw=1.5, label=f'tau_hat={alarm_t}')
axes[0].set_ylabel('x_t'); axes[0].legend(loc='upper right', fontsize=8)
axes[0].set_title('Наблюдаемый ряд')

# Panel 2 — latent norm ||Z_t||
Z_norm = np.linalg.norm(Z_det, axis=1)
axes[1].plot(t_ax, Z_norm, lw=0.8, color='mediumpurple', alpha=0.9, label='||Z_t||')
axes[1].axhline(np.sqrt(D), color='gray', ls=':', lw=1.2,
                label=f'E[||Z||] под H_0 = sqrt({D}) = {D**0.5:.2f}')
axes[1].axvline(TAU_DET, color='red', ls='--', lw=1.5)
if alarm_t:
    axes[1].axvline(alarm_t, color='green', ls=':', lw=1.5)
axes[1].set_ylabel('||Z_t||'); axes[1].legend(fontsize=8)
axes[1].set_title('Норма латентного кода (под H_0: E[||Z||^2] = d)')

# Panel 3 — MMD² sliding window
axes[2].plot(t_ax[W_MMD:], mmd_det[W_MMD:], lw=1.2, color='darkorange', label='MMD^2_t')
axes[2].axhline(h_mmd, color='red', ls='--', lw=1.5, label=f'h_mmd = {h_mmd:.5f}')
axes[2].axvline(TAU_DET, color='red', ls='--', lw=1.5, label=f'tau={TAU_DET}')
if alarm_t:
    axes[2].axvline(alarm_t, color='green', ls=':', lw=2,
                    label=f'tau_hat={alarm_t}  (задержка {delay})')
    axes[2].scatter([alarm_t], [mmd_det[alarm_t]], color='green', zorder=5, s=100)
axes[2].fill_between(t_ax[W_MMD:], h_mmd, mmd_det[W_MMD:],
                     where=mmd_det[W_MMD:] >= h_mmd, alpha=0.25, color='green')
axes[2].set_ylabel('MMD^2'); axes[2].set_xlabel('t')
axes[2].legend(fontsize=8)
axes[2].set_title('MMD^2 скользящего окна vs N(0,I)  (непараметрический, без предположений об H_1)')

plt.suptitle(f'DDIM + MMD тест  (w={W_MMD}, sigma={SIGMA_MMD:.2f}, alpha={ALPHA})',
             fontsize=12, y=1.01)
plt.tight_layout(); plt.show()

In [ ]:
torch.manual_seed(777)

N_TRIALS  = 50
TAU_FIXED = 100
L_TRIAL   = 180

delays       = []
false_alarms = 0
missed       = 0

for trial in range(N_TRIALS):
    x_tr = torch.cat([sample_p0(TAU_FIXED), sample_p1(L_TRIAL - TAU_FIXED)])

    Z_tr = np.zeros((L_TRIAL, D))
    for i in range(L_TRIAL):
        Z_tr[i] = ddim_encode(x_tr[i], h_fixed_kl).numpy()

    alarm = None
    for t in range(W_MMD, L_TRIAL):
        if mmd2_vs_gaussian(Z_tr[t - W_MMD:t]) >= h_mmd:
            alarm = t
            break

    if alarm is None:
        missed += 1
    elif alarm < TAU_FIXED:
        false_alarms += 1
    else:
        delays.append(alarm - TAU_FIXED)

    if (trial + 1) % 10 == 0:
        print(f"Trial {trial+1:3d}/{N_TRIALS}")

delays = np.array(delays)
print(f"\n=== Итоги ({N_TRIALS} испытаний, tau={TAU_FIXED}, w={W_MMD}, h_mmd={h_mmd:.5f}) ===")
print(f"  Обнаружено (tau_hat >= tau) : {len(delays)}/{N_TRIALS}  ({100*len(delays)/N_TRIALS:.0f}%)")
print(f"  Ложных тревог (tau_hat < tau): {false_alarms}/{N_TRIALS}  ({100*false_alarms/N_TRIALS:.0f}%)")
print(f"  Пропущено                   : {missed}/{N_TRIALS}")
if len(delays) > 0:
    print(f"  Задержка  median={np.median(delays):.1f},  mean={np.mean(delays):.1f} +/- {np.std(delays):.1f}")

if len(delays) > 0:
    plt.figure(figsize=(9, 3.5))
    plt.hist(delays, bins=range(0, int(delays.max()) + 5, 2),
             color='darkorange', alpha=0.75, edgecolor='k')
    plt.axvline(np.median(delays), color='red', ls='--', lw=2,
                label=f'Медиана = {np.median(delays):.0f}')
    plt.axvline(W_MMD, color='gray', ls=':', lw=1.5,
                label=f'Min задержка = w = {W_MMD}')
    plt.xlabel('Задержка обнаружения tau_hat - tau')
    plt.ylabel('Частота')
    plt.title(f'Распределение задержки (w={W_MMD}, h_mmd={h_mmd:.5f}, alpha={ALPHA})')
    plt.legend(); plt.tight_layout(); plt.show()

---
## Ремарка: как мы аппроксимируем порог

### Структура нулевого распределения MMD²

Под $H_0$ статистика имеет следующую асимптотику:

$$w\cdot\widehat{\mathrm{MMD}}^2 \;\xrightarrow{d}\; \sum_{l=1}^{\infty} \lambda_l\,(Z_l^2 - 1), \quad Z_l \sim \mathcal{N}(0,1) \text{ i.i.d.}$$

где $\lambda_l$ — собственные числа оператора ядра $T_k f(x) = \int k(x,y)\,f(y)\,dP_0(y)$ в $L^2(P_0)$. Для RBF-ядра с $\mathcal{N}(0,I)$ — это полиномы Эрмита с геометрически убывающими $\lambda_l \propto \left(\tfrac{\sigma^2}{\sigma^2+2}\right)^l$. Замкнутого CDF не существует.

---

### Почему используем бутстрэп из $\mathcal{N}(0,I)$

В реальной постановке у нас нет гарантированно доразладочного калибровочного набора данных: момент разладки $\tau$ случаен и неизвестен. Поэтому мы **не можем** строить нулевое распределение по реальным наблюдениям из потока.

Единственный выход — принять **предположение об идеальном деноизере**:

$$\mathrm{DDIM}(p_0) = \mathcal{N}(0,I) \text{ (точно)}$$

Тогда нулевое распределение $\widehat{\mathrm{MMD}}^2$ совпадает с распределением при сэмплировании из $\mathcal{N}(0,I)$, и порог симулируется без единой точки из потока:

```python
null_stats = [mmd2_vs_gaussian(rng.standard_normal((W_MMD, D))) for _ in range(3000)]
h_mmd = np.quantile(null_stats, 1 - ALPHA)
```

---

### Что происходит при несовершенном деноизере

Если $\mathrm{DDIM}(p_0) \neq \mathcal{N}(0,I)$ (что неизбежно при конечном обучении), то реальная статистика под $H_0$ **систематически выше** теоретической нулевой. Следствие: уровень ложных тревог превысит $\alpha$. Поэтому хорошее качество DDIM-кодирования — не просто желательное, а **необходимое** условие корректности теста.

---

### Сравнение альтернатив

| | Бутстрэп из $\mathcal{N}(0,I)$ | Бутстрэп из $Z_\text{cal}$ | Гамма-аппроксимация |
|---|---|---|---|
| Нужны данные из потока | ❌ | ✅ ($N_\text{cal} \gg w$) | ❌ |
| Учитывает несовершенство DDIM | ❌ | ✅ | ❌ |
| Корректен при идеальном деноизере | ✅ | ✅ | ⚠️ |
| Применим без офлайн-набора | ✅ | ❌ | ✅ |

Мы используем первый столбец: предположение об идеальности оправдано диагностическими графиками DDIM(p₀).